# Code pour nettoyer les fichiers Gallica

In [5]:
import os
import re

# -----------------------------
# PARAMÈTRES
# -----------------------------

DOSSIER_ENTREE = "txt_bruts"
DOSSIER_SORTIE = "txt_nettoyes"

os.makedirs(DOSSIER_SORTIE, exist_ok=True)

# -----------------------------
# FONCTIONS DE DÉTECTION
# -----------------------------

def est_separateur(ligne):
    return bool(re.fullmatch(r"-{5,}", ligne.strip()))

def est_numero_page(ligne):
    return bool(re.fullmatch(r"\d{1,4}", ligne.strip()))

def est_bruit_ocr(ligne):
    # beaucoup de symboles non alphabétiques
    return bool(re.search(r"[§*^%<>_=]{2,}", ligne))

def est_titre_majuscules(ligne):
    txt = ligne.strip()
    if len(txt) < 4:
        return False
    lettres = re.findall(r"[A-ZÀ-ÖØ-Ý]", txt)
    return len(lettres) > 0 and len(lettres) / max(len(txt), 1) > 0.6

def est_bruit_court(ligne):
    return len(ligne.strip()) <= 2

def est_debut_texte(ligne):
    # phrase assez longue contenant des minuscules
    return len(ligne.strip()) > 40 and re.search(r"[a-zà-öø-ÿ]", ligne)

# -----------------------------
# RECOMPOSITION DES PARAGRAPHES
# -----------------------------

def recomposer_paragraphes(lignes):
    paragraphes = []
    courant = ""

    for ligne in lignes:
        l = ligne.strip()

        if l == "":
            if courant:
                paragraphes.append(courant.strip())
                courant = ""
            continue

        # si la ligne se termine par une ponctuation forte, on ferme le paragraphe
        if courant:
            if not courant.endswith(("-", "—")):
                courant += " "
            courant += l
        else:
            courant = l

        if re.search(r"[.!?…]$", l):
            paragraphes.append(courant.strip())
            courant = ""

    if courant:
        paragraphes.append(courant.strip())

    return "\n\n".join(paragraphes)

# -----------------------------
# TRAITEMENT DES FICHIERS
# -----------------------------

for nom_fichier in os.listdir(DOSSIER_ENTREE):
    if not nom_fichier.lower().endswith(".txt"):
        continue

    chemin_entree = os.path.join(DOSSIER_ENTREE, nom_fichier)
    chemin_sortie = os.path.join(DOSSIER_SORTIE, nom_fichier)

    with open(chemin_entree, "r", encoding="utf-8", errors="ignore") as f:
        lignes = f.readlines()

    # 1. suppression du paratexte initial
    debut_trouve = False
    corps = []

    for ligne in lignes:
        if not debut_trouve:
            if est_debut_texte(ligne):
                debut_trouve = True
            else:
                continue
        corps.append(ligne)

    # 2. nettoyage ligne par ligne
    nettoyees = []
    for ligne in corps:

        if est_separateur(ligne):
            continue
        if est_numero_page(ligne):
            continue
        if est_bruit_ocr(ligne):
            continue
        if est_bruit_court(ligne):
            continue
        if est_titre_majuscules(ligne):
            continue

        nettoyees.append(ligne)

    # 3. recomposition des paragraphes
    texte_final = recomposer_paragraphes(nettoyees)

    # 4. écriture du fichier nettoyé
    with open(chemin_sortie, "w", encoding="utf-8") as f:
        f.write(texte_final)

    print("Nettoyé :", nom_fichier)

print("Traitement terminé.")


Nettoyé : Les bêtes en robe de chambe.txt
Nettoyé : 1857_Meurice_Les-tyrans-de-village-suivi-de-l-Ecole-des-proprietaires_gal.txt
Nettoyé : Récits de terroir.txt
Nettoyé : L'histoire naturelle en action 1e edition.txt
Nettoyé : 1854_Meurice_La-Famille-Aubry-Vol3_gal.txt
Nettoyé : Des chiens anglais de chasse et de.txt
Nettoyé : 1854_Meurice_La-Famille-Aubry-Vol2_gal.txt
Nettoyé : Nouveaux contes d'un coureur des bois.txt
Nettoyé : Oiseaux de chasse 2e édition.txt
Nettoyé : 1854-1862_Maquet_La-belle-Gabrielle-La-Maison-du-baigneur-Dettes-de-coeur-Les-vertes-feuilles_Coffret-bnf.txt
Nettoyé : 1854_Meurice_La-Famille-Aubry-Vol1_gal.txt
Nettoyé : 1869_Meurice_Les-Chevaliers-de-l-esprit-Cesara_gal.txt
Nettoyé : Le monde des champs.txt
Nettoyé : Histoire d'un trop bon chien.txt
Nettoyé : Bêtes et gens.txt
Nettoyé : La vie à la campagne.txt
Nettoyé : Les chiens d'arrêt français et anglais.txt
Nettoyé : L'histoire naturelle en action 2e edition.txt
Nettoyé : Une chasse d'écolier.txt
Nettoyé : 